# K-FRAG protocol v1 demo

This notebook demonstrates the non-neural provenance protocol. The secret key is used only in memory and is never displayed.

In [ ]:
import random

from kfrag.crypto.authentication import verify_tag
from kfrag.crypto.packets import create_packets, verify_and_recover_token
from kfrag.crypto.token import ProvenanceToken

In [ ]:
secret_key = b"notebook-demo-key-do-not-use-in-production"
token = ProvenanceToken.generate(issuer_id=42, version=1)
packets = create_packets(token, secret_key)

print(f"Issuer ID: {token.issuer_id}")
print(f"Asset ID: {token.asset_id}")
print(f"Protocol version: {token.version}")
print(f"Authenticated packets produced: {len(packets)}")

In [ ]:
removed_indices = set(random.sample(range(len(packets)), 4))
surviving_packets = [
    packet for packet in packets if packet.region_index not in removed_indices
]

print(f"Removed region indices: {sorted(removed_indices)}")
print(f"Surviving packets: {len(surviving_packets)}")

In [ ]:
recovered_token = verify_and_recover_token(surviving_packets, secret_key)
assert recovered_token == token

tag_results = [
    verify_tag(
        secret_key,
        recovered_token.pack(),
        packet.region_index,
        packet.coded_symbol,
        packet.authentication_tag,
    )
    for packet in surviving_packets
]
assert len(tag_results) == 12 and all(tag_results)

print(f"Recovered token matches original: {recovered_token == token}")
print(f"All surviving tags verified: {all(tag_results)}")

In [ ]:
wrong_key_rejected = False
try:
    verify_and_recover_token(surviving_packets, b"definitely-the-wrong-key")
except ValueError:
    wrong_key_rejected = True

assert wrong_key_rejected
print(f"Wrong key rejected: {wrong_key_rejected}")